# Workload placement demo — student guide

This notebook recommends where one new application should run. **Gurobi** solves the optimization model. In AI mode, **OptiMind** first proposes a structured JSON model (IR); trusted code validates it before accepting a recommendation. Generated Python is never executed.

## Run order: start with the reference demonstration

1. Run **Step 1 — Mount Drive** and **Step 2 — Stage project** once.
2. In **Step 3 — Choose mode**, keep `USE_AI = False`, `RUN_DEMO = False`, and `ALLOW_REFERENCE_FALLBACK = False`.
3. Run **Steps 4, 5 and 6**: dependency setup, runtime verification, and review.
4. If Step 6 passes, change only Step 7 to `RUN_DEMO = True` and run it.
5. Run **Step 8 — FINAL DEMO CELL** once. Expect source `REFERENCE_FALLBACK`. This explicitly selected mode does not invoke OptiMind.

## Then try AI in the same session

1. After the reference run finishes, change **Step 3** to `USE_AI = True`; keep its other flags False. Run Step 3.
2. Run **Steps 4, 5 and 6 again**, in order. AI dependencies must be installed before the new runtime verification.
3. If Step 6 passes and you approve model inference, set/run `RUN_DEMO = True` in **Step 7**, then run **Step 8** once.
4. Keep fallback False to see whether the AI model passes on its own. AI may fail safely; a valid JSON response is not proof of mathematical correctness.

**Do not rerun Steps 1–2 just to switch mode.** New staging creates a different working directory and cache. Do not use Run all after enabling Step 7, and never start a second inference while one is running. If the Colab session expires, start again from Step 1 and obtain fresh verification. Preserve all previous Drive results.

Flags: `USE_AI` chooses the method; `RUN_DEMO` authorizes one launch; `ALLOW_REFERENCE_FALLBACK` permits a trusted reference result after AI failure. They have different purposes.


## Step 1 — Mount Google Drive
Run once. Approve Colab’s Drive access dialog. This makes your uploaded project and persistent results available.


In [ ]:
# Connect persistent storage; this does not start optimization.
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Stage the project and install baseline dependencies
Run once per session. Check DRIVE_PROJECT matches your uploaded folder. The unique `/content` directory is temporary; results are later copied back to Drive. Existing Drive results are excluded from staging and are not deleted. Keep the model cache on local Colab storage for performance.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, uuid
DRIVE_PROJECT = Path('/content/drive/MyDrive/workload-placement-research')
USE_AI = False  # True invokes the pinned OptiMind checkpoint; requires approval.
RUN_DEMO = False  # Explicit opt-in after verification; consumed on launch.
ALLOW_REFERENCE_FALLBACK = False  # AI failure stays a failure unless explicitly enabled.
HF_TOKEN_SECRET_NAME = 'HF_TOKEN'
# UUID creates a new working directory without overwriting earlier evidence.
RUNTIME_PROJECT = Path('/content') / ('placement-demo-' + uuid.uuid4().hex[:12])
# Copy code and inputs, excluding old results, environments and model weights.
shutil.copytree(DRIVE_PROJECT, RUNTIME_PROJECT, ignore=shutil.ignore_patterns('results', '.venv', '.git', '__pycache__', '.pytest_cache', '.model_cache'))
os.chdir(RUNTIME_PROJECT)
sys.path.insert(0, str(RUNTIME_PROJECT))
RESULTS = RUNTIME_PROJECT / 'results'
CACHE = RUNTIME_PROJECT / '.model_cache'
print('Python:', sys.executable, 'Project:', RUNTIME_PROJECT)
# sys.executable ensures pip uses the same Python as this notebook.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


## Step 3 — Choose the method (edit this cell to switch to AI)

For your first run leave USE_AI False. After the reference run, change it to True here and rerun Steps 4–6. Running this cell resets launch permission to False. Do not edit the setup cell to switch methods.


In [ ]:
USE_AI = False  # First reference run: False. Later approved AI run: True.
RUN_DEMO = False  # Preparation never starts a model; Step 7 grants one launch.
ALLOW_REFERENCE_FALLBACK = False  # Keep False to expose AI errors rather than substitute a reference result.
print('Selected mode:', 'AI_ASSISTED' if USE_AI else 'REFERENCE_FALLBACK')


## Step 4 — Prepare dependencies and optional Hugging Face authentication

Run after choosing the mode. In reference mode this cell skips AI installation. In AI mode, add a Colab Secret named **HF_TOKEN** (key icon in the sidebar) and grant notebook access. Never paste the token into code or share it. Authentication can reduce anonymous download throttling; it does not guarantee downloads succeed.

This cell removes unnecessary torchvision, installs the pinned text stack and checks imports/CUDA. It does not download model weights or start inference.


In [ ]:
# Dependency setup only; no model download or inference here.
if USE_AI:
    from google.colab import userdata
    if not os.environ.get('HF_TOKEN'):
        try:
            os.environ['HF_TOKEN'] = userdata.get(HF_TOKEN_SECRET_NAME)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            print('HF_TOKEN unavailable; configure Colab Secrets to reduce anonymous throttling.')
    print('HF authentication configured:', bool(os.environ.get('HF_TOKEN')))
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-inference-transformers.txt', '-c', 'requirements.txt'], check=True)
    # Save an import/CUDA report for the review cell; model_loaded stays False.
    subprocess.run([sys.executable, '-m', 'src.text_environment', '--output', str(RESULTS / 'text_environment.json')], check=True)
else:
    print('Reference mode: no AI dependency setup needed.')


## Step 5 — Verify this runtime and synchronize evidence

Run after Step 4, including when switching to AI. Tests, benchmark audits, restricted Gurobi licensing and real Drive synchronization must pass. Each run uses a fresh verification folder. `check=True` stops on failure; `finally` preserves available diagnostics in Drive even on failure. Do not bypass an error or edit a readiness flag.


In [ ]:
# Existing runtime safety gates: full suite, license, benchmark audits and real Drive sync.
from src.runtime_verification import sync_verified
VERIFICATION_ROOT = RESULTS / ('verification-' + uuid.uuid4().hex[:12])
try:
    subprocess.run([sys.executable, '-m', 'src.runtime_verification',
                    '--output', str(VERIFICATION_ROOT),
                    '--drive-output', str(DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name / VERIFICATION_ROOT.name)], check=True)
finally:
    if RESULTS.exists(): sync_verified(RESULTS, DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name)
RUNTIME_EVIDENCE = next(VERIFICATION_ROOT.glob('runtime-*'))
assert json.loads((RUNTIME_EVIDENCE / 'runtime.json').read_text())['ready_for_inference']


## Step 6 — REVIEW VERIFICATION (no inference)

Run this cell and read the output. Required: VERIFIED, Ready=True, no readiness reasons, all expected tests passed and zero failures/skips. The expected count comes from the uploaded test manifest, so it can increase as tests are added.

For AI also require a passing text-environment report, STRUCTURED_IR, and the preserved checkpoint. Review the displayed output-token budget (currently 8192). These checks authorize an attempt; they do not predict a correct generated model. If this cell fails, stop before Steps 7–8.


In [ ]:
# Read persisted evidence; do not edit reports or create a model here.
import yaml
report = json.loads((RUNTIME_EVIDENCE / 'runtime.json').read_text())
tests = report.get('tests', {})
expected_count = len(json.loads((RUNTIME_PROJECT / 'tests/suite_manifest.json').read_text())['nodeids'])
print('Mode:', 'AI_ASSISTED' if USE_AI else 'REFERENCE_FALLBACK')
print('Fallback after AI failure:', ALLOW_REFERENCE_FALLBACK)
print('Report:', RUNTIME_EVIDENCE)
print('Status:', report['colab_verification'], 'Ready:', report['ready_for_inference'])
print('Reasons:', report.get('readiness_reasons', []))
print('Tests:', {key: tests.get(key) for key in ('collected', 'passed', 'failed', 'skipped')})
assert report['colab_verification'] == 'VERIFIED' and report['ready_for_inference'] is True
assert not report.get('readiness_reasons', [])
assert tests.get('collected') == tests.get('passed') == expected_count
assert tests.get('failed') == 0 and tests.get('skipped') == 0
if USE_AI:
    environment_path = RESULTS / 'text_environment.json'
    assert environment_path.is_file(), 'Run Step 4 with USE_AI=True first.'
    environment = json.loads(environment_path.read_text())
    print('Text environment passed:', environment['passed'], 'Reasons:', environment.get('reasons', []))
    assert environment['passed'] and environment['imports_passed'] and environment['cuda_available']
    settings = yaml.safe_load((RUNTIME_PROJECT / 'tests/fixtures/config/demo/optimind.yaml').read_text())
    print('Output budget:', settings['inference']['max_new_tokens'])
    print('Generation mode:', settings['generation']['mode'])
    print('Checkpoint:', settings['model']['revision'])
    assert settings['generation']['mode'] == 'STRUCTURED_IR'
    assert settings['model']['revision'] == '817e06666a39155f9352d00354b3227d230da46d'
print('Preparation checks passed. No demo started. Review before enabling Step 7.')


## Step 7 — Approve one launch

Only after Step 6 passes, change the line below to **RUN_DEMO = True** and run this cell. This is your explicit approval; there is no additional approval button. The cell only sets a flag. Step 8 performs the work.

The saved default is False so a fresh notebook cannot accidentally start inference. For AI, confirm USE_AI=True and that Steps 4–6 were rerun after the switch.


In [ ]:
RUN_DEMO = False  # Change to True only after reviewing Step 6, then run this cell.
print('Launch authorized:', RUN_DEMO)


## Step 8 — FINAL DEMO CELL (run once)

Run this entire cell after Step 7. It consumes the permission by resetting RUN_DEMO=False: **leave that reset unchanged**. Existing worker detection blocks overlap. Each launch writes a new configuration and result directory, then synchronizes evidence to Drive.

Reference mode runs Gurobi only. AI mode may download/load the model and generate IR before validation/solving. Do not interrupt or launch again because it takes time. Only this new launch’s results are printed.


In [ ]:
if not RUN_DEMO:
    print('Prepared only. Review verification; set RUN_DEMO=True in a new cell to launch once.')
else:
    RUN_DEMO = False  # Consume approval; do not change this reset.
    # Reject overlap, including orphaned inference workers.
    for process in Path('/proc').iterdir():
        if process.name.isdigit():
            try:
                args = (process / 'cmdline').read_bytes().split(b'\0')
                assert not any(m in args for m in (b'src.inference_worker', b'src.demo', b'tests.support.optimind_experiment')), 'Existing worker/controller: '+process.name
            except (FileNotFoundError, ProcessLookupError):
                pass
    import yaml
    # Select an existing YAML; bank policies remain external to notebook code.
    template = Path('tests/fixtures/config/demo/ai_single.yaml' if USE_AI else 'tests/fixtures/config/demo/single.yaml')
    settings = yaml.safe_load(template.read_text())
    settings['allow_reference_fallback'] = ALLOW_REFERENCE_FALLBACK
    # Never overwrite a previous run. Save the actual configuration used.
    launch = RESULTS / ('launch-' + uuid.uuid4().hex[:12])
    launch.mkdir(parents=True, exist_ok=False)
    config = launch / 'demo.yaml'
    config.write_text(yaml.safe_dump(settings, sort_keys=False))
    print('Mode:', settings['mode'], 'Fallback enabled:', ALLOW_REFERENCE_FALLBACK)
    # The CLI runs trusted code. Model-generated Python is never executed.
    try:
        completed = subprocess.run([sys.executable, '-m', 'src.demo', '--config', str(config),
                                    '--output', str(launch / 'runs'), '--runtime-evidence', str(RUNTIME_EVIDENCE),
                                    '--cache-dir', str(CACHE)])
        print('Exit:', completed.returncode)
    finally:
        # Preserve failures as well as successes, verifying file hashes in Drive.
        sync_verified(RESULTS, DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name)
    for path in (launch / 'runs').glob('*/recommendation.json'):
        result = json.loads(path.read_text())
        print(path)
        print(json.dumps({k: result[k] for k in ('status', 'source', 'fallback_used', 'fallback_reason', 'placement', 'metrics', 'errors')}, indent=2))


## Reading the result and finding files

- **RECOMMENDED / OPTIMIND_IR**: generated algebra passed the demo’s conservative gates and the placement was independently validated.
- **RECOMMENDED / REFERENCE_FALLBACK**: the trusted reference produced the placement. In explicit reference mode fallback_used=True is expected even though no AI failed.
- **FAILED / REJECTED**: inspect errors and saved AI/solver diagnostics; do not manually repair a completed report or repeatedly launch.
- **INFEASIBLE**: the solver reported infeasibility; no placement is invented.

Open the printed recommendation.json path. The Drive copy is under `MyDrive/workload-placement-research/results/<placement-demo-id>/<launch-id>/runs/<demo-id>/`. Also inspect summary.md, explanation.json, validation results and logs. Source/fallback flags matter even when Exit is 0. Keep prior outputs and completion hashes unchanged.


## Step 9 — Explain this result (read only)

Run after Step 8 in the same session. This only reads existing artifacts; it never starts inference or optimization. Missing AI files are expected for reference mode. Failed attempts may not have later validation or solver files.


In [ ]:
# Read existing results from the most recent Step 8 launch; no inference.
assert "launch" in globals(), "Run this in the session where Step 8 completed."
recommendations = sorted((launch / "runs").glob("*/recommendation.json"))
assert recommendations, "No recommendation found for this launch; inspect Step 8 diagnostics."
for recommendation in recommendations:
    run_folder = recommendation.parent
    print("RUN:", run_folder)
    print("INPUT SNAPSHOTS:", run_folder / "input_snapshot")
    for relative in (
        "recommendation.json",
        "ai/specification.json",
        "ai/compilation_provenance.json",
        "ai/spec_validation.json",
        "ai/validation/compiled/solver.json",
        "explanation.json",
    ):
        artifact = run_folder / relative
        print("\n---", relative, "---")
        print(artifact.read_text() if artifact.is_file()
              else "Not produced for this run; reference mode has no AI artifacts.")
    for relative in ("ai/raw_response.txt", "ai/generation.json", "ai/expanded_ir.json",
                     "ai/generated_ir.json", "ai/validation/validation.json",
                     "ai/validation/compiled/compiled.lp", "reference"):
        artifact = run_folder / relative
        if artifact.exists():
            print("Additional evidence:", artifact)
